# Routing


This notebook refactors the original LangChain + RunnableBranch routing example into a Flyte 2.0 task that uses LangGraph for the routing logic. We keep the same three logical handlers (`booking_handler`, `info_handler`, `unclear_handler`), but:

- Use **LangGraph** to implement the router and delegation graph.
- Use **Flyte 2** to wrap the routing logic in async tasks (`route_request`, `route_requests_batch`).
- Make it easy to run both locally (in the notebook) and remotely via Flyte.


In [ ]:
import os
from typing import TypedDict, Literal

import asyncio
import flyte
from flyte import TaskEnvironment, Resources, Secret

from langgraph.graph import StateGraph, END
from openai import AsyncOpenAI


# --- Flyte TaskEnvironment configuration ---

env = TaskEnvironment(
    name="routing_env",
    resources=Resources(cpu="1", memory="1Gi"),
    cache="auto",
    # In remote Flyte runs, this will mount the tenant secret as OPENAI_API_KEY.
    secrets=[Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY")],
)


# --- LangGraph state and graph definition ---

class RouterState(TypedDict, total=False):
    request: str
    decision: Literal["booker", "info", "unclear"]
    output: str


# Single OpenAI async client shared within the container.
client = AsyncOpenAI()  # reads OPENAI_API_KEY from the environment


async def router_node(state: RouterState) -> RouterState:
    """Call the LLM to decide how to route the request."""
    request = state["request"]
    messages = [
        {
            "role": "system",
            "content": (
                "Analyze the user's request and determine which specialist handler should process it.\n"
                "- If the request is related to booking flights or hotels, output 'booker'.\n"
                "- For general information questions, output 'info'.\n"
                "- If the request is unclear or doesn't fit either category, output 'unclear'.\n"
                "ONLY output one word: 'booker', 'info', or 'unclear'."
            ),
        },
        {"role": "user", "content": request},
    ]
    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=messages,
    )
    decision = (resp.choices[0].message.content or "").strip().lower()
    if decision not in ("booker", "info", "unclear"):
        decision = "unclear"
    # Preserve existing fields and add decision
    return {**state, "decision": decision}


def booking_handler(request: str) -> str:
    """Simulates the Booking Agent handling a request."""
    return f"Booking Handler processed request: '{request}'. Result: Simulated booking action."


def info_handler(request: str) -> str:
    """Simulates the Info Agent handling a request."""
    return f"Info Handler processed request: '{request}'. Result: Simulated information retrieval."


def unclear_handler(request: str) -> str:
    """Handles requests that couldn't be delegated."""
    return f"Coordinator could not delegate request: '{request}'. Please clarify."


async def booking_node(state: RouterState) -> RouterState:
    return {**state, "output": booking_handler(state["request"])}


async def info_node(state: RouterState) -> RouterState:
    return {**state, "output": info_handler(state["request"])}


async def unclear_node(state: RouterState) -> RouterState:
    return {**state, "output": unclear_handler(state["request"])}


def route_decision(state: RouterState) -> Literal["booker", "info", "unclear"]:
    return state["decision"]


graph = StateGraph(RouterState)

# Nodes
graph.add_node("router", router_node)
graph.add_node("booker", booking_node)
graph.add_node("info", info_node)
graph.add_node("unclear", unclear_node)

# Entry point and conditional edges
graph.set_entry_point("router")
graph.add_conditional_edges(
    "router",
    route_decision,
    {
        "booker": "booker",
        "info": "info",
        "unclear": "unclear",
    },
)

# Terminal edges
graph.add_edge("booker", END)
graph.add_edge("info", END)
graph.add_edge("unclear", END)

# Compile the LangGraph app
app = graph.compile()


# --- Flyte tasks wrapping the LangGraph app ---

@env.task
async def route_request(request: str) -> str:
    """Route a single request through the LangGraph app and return the handler output."""
    final_state = await app.ainvoke({"request": request})
    return final_state["output"]


@env.task
async def route_requests_batch(requests: list[str]) -> list[str]:
    """Route a batch of requests in parallel with bounded concurrency."""
    sem = asyncio.Semaphore(20)

    async def _one(req: str) -> str:
        async with sem:
            return await route_request(req)

    tasks = [_one(r) for r in requests]
    return list(await asyncio.gather(*tasks))


# --- Local testing helper (not run by Flyte) ---

async def _demo() -> None:
    examples = [
        "Book me a flight to London.",
        "What is the capital of Italy?",
        "Tell me about quantum physics.",
    ]
    for req in examples:
        out = await route_request(req)
        print(f"Request: {req}\nOutput:  {out}\n")


if __name__ == "__main__":
    if "OPENAI_API_KEY" not in os.environ:
        print("Set OPENAI_API_KEY in your environment before running this demo.")
    else:
        asyncio.run(_demo())
